# **STOCK MARKET ANALYST MODEL USING SVM / SVR**

In [ ]:
# =========================================================
# STOCK MARKET ANALYST MODEL USING SVM / SVR
# Predict next-day closing price from previous market data
# =========================================================

!pip install kaggle plotly -q

import os
import zipfile
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from google.colab import files
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# =========================
# DOWNLOAD DATASET FROM KAGGLE
# =========================

files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d jacksoncrow/stock-market-dataset

Saving symbols_valid_meta.csv to symbols_valid_meta (1).csv
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset
License(s): CC0-1.0
stock-market-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# =========================
# EXTRACT DATASET
# =========================

with zipfile.ZipFile("stock-market-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("stock_market_dataset")

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [ ]:
# =========================
# SELECT STOCK TICKER
# =========================

TICKER = "AMZN"  # coba ganti ke TSLA, MSFT, AMZN, GOOGL, NVDA, META

csv_path = f"stock_market_dataset/stocks/{TICKER}.csv"

df = pd.read_csv(csv_path)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1,1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
2,1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
3,1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
4,1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [ ]:
# =========================
# BASIC EDA
# =========================

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe())

Dataset shape: (5758, 7)

Missing values:
Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Summary statistics:


,Date,Open,High,Low,Close,Adj Close,Volume
count,5758,5758.000000,5758.000000,5758.000000,5758.000000,5758.000000,5.758000e+03
mean,2008-10-22 21:30:41.889544960,340.458153,344.156408,336.344390,340.417580,340.417580,7.556094e+06
min,1997-05-15 00:00:00,1.406250,1.447917,1.312500,1.395833,1.395833,4.872000e+05
25%,2003-02-05 06:00:00,37.460001,38.334999,36.812499,37.562500,37.562500,3.685525e+06
50%,2008-10-22 12:00:00,81.965000,83.520000,79.875000,81.599998,81.599998,5.692450e+06
75%,2014-07-14 18:00:00,335.267494,337.537491,331.727501,334.290001,334.290001,8.594350e+06
max,2020-04-01 00:00:00,2173.070068,2185.949951,2161.120117,2170.219971,2170.219971,1.043292e+08
std,NaN,523.365374,528.138556,517.726971,523.140207,523.140207,7.325904e+06


In [ ]:
# =========================
# INTERACTIVE CANDLESTICK CHART
# =========================

fig = go.Figure(data=[
    go.Candlestick(
        x=df["Date"],
        open=df["Open"],
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        name=TICKER
    )
])

fig.update_layout(
    title=f"{TICKER} Candlestick Chart",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# =========================
# FEATURE ENGINEERING
# =========================
# Target:
# Predict tomorrow's Close price using today's market information.
#
# Features:
# Open, High, Low, Close, Volume
# plus technical features from historical data.

df_model = df.copy()

df_model["Daily_Return"] = df_model["Close"].pct_change()
df_model["Price_Range"] = df_model["High"] - df_model["Low"]
df_model["Body_Size"] = abs(df_model["Close"] - df_model["Open"])

df_model["MA_5"] = df_model["Close"].rolling(window=5).mean()
df_model["MA_10"] = df_model["Close"].rolling(window=10).mean()
df_model["MA_20"] = df_model["Close"].rolling(window=20).mean()

df_model["Volatility_5"] = df_model["Daily_Return"].rolling(window=5).std()
df_model["Volatility_10"] = df_model["Daily_Return"].rolling(window=10).std()

df_model["Volume_MA_5"] = df_model["Volume"].rolling(window=5).mean()
df_model["Volume_MA_10"] = df_model["Volume"].rolling(window=10).mean()

# Target: next day's closing price
df_model["Target_Next_Close"] = df_model["Close"].shift(-1)

df_model = df_model.dropna().reset_index(drop=True)

features = [
    "Open", "High", "Low", "Close", "Volume",
    "Daily_Return", "Price_Range", "Body_Size",
    "MA_5", "MA_10", "MA_20",
    "Volatility_5", "Volatility_10",
    "Volume_MA_5", "Volume_MA_10"
]

X = df_model[features]
y = df_model["Target_Next_Close"]

df_model.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Daily_Return,Price_Range,Body_Size,MA_5,MA_10,MA_20,Volatility_5,Volatility_10,Volume_MA_5,Volume_MA_10,Target_Next_Close
0,1997-06-12,1.583333,1.645833,1.552083,1.604167,1.604167,1632000,0.040541,0.093750,0.020833,1.614583,1.552083,1.574740,0.053873,0.048912,3687600.0,3156000.0,1.583333
1,1997-06-13,1.625000,1.625000,1.583333,1.583333,1.583333,693600,-0.012987,0.041667,0.041667,1.600000,1.560417,1.555990,0.039764,0.049239,2264880.0,2965920.0,1.572917
2,1997-06-16,1.604167,1.604167,1.562500,1.572917,1.572917,913200,-0.006579,0.041667,0.031250,1.577083,1.566667,1.548177,0.036942,0.049411,1977120.0,2998080.0,1.505208
3,1997-06-17,1.598958,1.598958,1.494792,1.505208,1.505208,4706400,-0.043046,0.104167,0.093750,1.561458,1.569271,1.538021,0.031356,0.051184,1826640.0,3350400.0,1.510417
4,1997-06-18,1.520833,1.536458,1.500000,1.510417,1.510417,2464800,0.003460,0.036458,0.010417,1.555208,1.578646,1.531771,0.030212,0.048682,2082000.0,3288840.0,1.510417


In [ ]:
# =========================
# TRAIN TEST SPLIT
# =========================
# Time series data should not be randomly shuffled.
# First 80% = training data
# Last 20% = testing data

split_index = int(len(df_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

date_test = df_model["Date"].iloc[split_index:]

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

Training data: (4590, 15)
Testing data : (1148, 15)


In [ ]:
# =========================
# SVM MODEL WITH HYPERPARAMETER TUNING
# =========================

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

param_grid = {
    "svr__kernel": ["rbf"],
    "svr__C": [10, 100, 500],
    "svr__gamma": ["scale", 0.01, 0.001],
    "svr__epsilon": [0.01, 0.1, 0.5]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'svr__C': 500, 'svr__epsilon': 0.1, 'svr__gamma': 0.001, 'svr__kernel': 'rbf'}


In [ ]:
# =========================
# PREDICTION
# =========================

y_pred = best_model.predict(X_test)

results = pd.DataFrame({
    "Date": date_test.values,
    "Actual_Next_Close": y_test.values,
    "Predicted_Next_Close": y_pred,
    "Previous_Close": X_test["Close"].values
})

results["Actual_Return"] = (results["Actual_Next_Close"] - results["Previous_Close"]) / results["Previous_Close"]
results["Predicted_Return"] = (results["Predicted_Next_Close"] - results["Previous_Close"]) / results["Previous_Close"]

results.head()

,Date,Actual_Next_Close,Predicted_Next_Close,Previous_Close,Actual_Return,Predicted_Return
0,2015-09-09,522.239990,511.006533,516.890015,0.010350,-0.011382
1,2015-09-10,529.440002,513.058708,522.239990,0.013787,-0.017581
2,2015-09-11,521.380005,518.878768,529.440002,-0.015224,-0.019948
3,2015-09-14,522.369995,513.171023,521.380005,0.001899,-0.015745
4,2015-09-15,527.390015,514.326843,522.369995,0.009610,-0.015397


In [ ]:
# =========================
# MODEL EVALUATION
# =========================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Baseline model:
# simple prediction that tomorrow's close = today's close
baseline_pred = X_test["Close"].values

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

# Direction accuracy:
# whether the model correctly predicts price movement direction
actual_direction = np.sign(results["Actual_Next_Close"] - results["Previous_Close"])
predicted_direction = np.sign(results["Predicted_Next_Close"] - results["Previous_Close"])

direction_accuracy = (actual_direction == predicted_direction).mean() * 100

evaluation = pd.DataFrame({
    "Model": ["SVR Model", "Baseline: Tomorrow = Today"],
    "MAE": [mae, baseline_mae],
    "RMSE": [rmse, baseline_rmse],
    "R2 Score": [r2, baseline_r2]
})

display(evaluation)

print(f"Direction Accuracy: {direction_accuracy:.2f}%")

,Model,MAE,RMSE,R2 Score
0,SVR Model,564.590658,762.215080,-1.315248
1,Baseline: Tomorrow = Today,16.192011,25.755985,0.997356


Direction Accuracy: 44.60%


In [ ]:
# =========================
# VISUALIZE ACTUAL VS PREDICTED
# =========================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results["Date"],
    y=results["Actual_Next_Close"],
    mode="lines",
    name="Actual Next Close"
))

fig.add_trace(go.Scatter(
    x=results["Date"],
    y=results["Predicted_Next_Close"],
    mode="lines",
    name="Predicted Next Close"
))

fig.update_layout(
    title=f"{TICKER} Next-Day Close Price Prediction Using SVR",
    xaxis_title="Date",
    yaxis_title="Close Price",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# =========================
# ERROR VISUALIZATION
# =========================

results["Prediction_Error"] = results["Actual_Next_Close"] - results["Predicted_Next_Close"]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results["Date"],
    y=results["Prediction_Error"],
    mode="lines",
    name="Prediction Error"
))

fig.update_layout(
    title=f"{TICKER} Prediction Error Over Time",
    xaxis_title="Date",
    yaxis_title="Actual - Predicted",
    template="plotly_white",
    height=450
)

fig.show()

In [ ]:
# =========================
# NEXT DAY PRICE PREDICTION
# =========================

latest_data = df_model[features].iloc[-1:]
latest_close = df_model["Close"].iloc[-1]
latest_date = df_model["Date"].iloc[-1]

next_close_prediction = best_model.predict(latest_data)[0]
predicted_change = next_close_prediction - latest_close
predicted_change_pct = (predicted_change / latest_close) * 100

if predicted_change_pct > 1:
    signal = "BUY"
elif predicted_change_pct < -1:
    signal = "SELL"
else:
    signal = "HOLD"

print("========== STOCK MARKET ANALYST RESULT ==========")
print(f"Ticker                 : {TICKER}")
print(f"Latest Date            : {latest_date}")
print(f"Latest Close Price     : {latest_close:.2f}")
print(f"Predicted Next Close   : {next_close_prediction:.2f}")
print(f"Predicted Change       : {predicted_change:.2f}")
print(f"Predicted Change (%)   : {predicted_change_pct:.2f}%")
print(f"Analyst Signal         : {signal}")
print("=================================================")

========== STOCK MARKET ANALYST RESULT ==========
Ticker                 : AMZN
Latest Date            : 2020-03-31 00:00:00
Latest Close Price     : 1949.72
Predicted Next Close   : 629.14
Predicted Change       : -1320.58
Predicted Change (%)   : -67.73%
Analyst Signal         : SELL


In [ ]:
# =========================
# FINAL SUMMARY FOR REPORT
# =========================

print(f"""
Project Summary

This project builds a stock market analyst model using Support Vector Regression.
The model predicts the next-day closing price of {TICKER} stock based on previous market data.

Input features:
- Open price
- High price
- Low price
- Close price
- Trading volume
- Daily return
- Price range
- Moving averages
- Historical volatility
- Volume moving averages

Target:
- Next-day closing price

Best SVR Parameters:
{grid_search.best_params_}

Model Performance:
- MAE  : {mae:.4f}
- RMSE : {rmse:.4f}
- R2   : {r2:.4f}
- Direction Accuracy: {direction_accuracy:.2f}%

Latest Prediction:
- Latest close price: {latest_close:.2f}
- Predicted next close price: {next_close_prediction:.2f}
- Analyst signal: {signal}
""")


Project Summary

This project builds a stock market analyst model using Support Vector Regression.
The model predicts the next-day closing price of AMZN stock based on previous market data.

Input features:
- Open price
- High price
- Low price
- Close price
- Trading volume
- Daily return
- Price range
- Moving averages
- Historical volatility
- Volume moving averages

Target:
- Next-day closing price

Best SVR Parameters:
{'svr__C': 500, 'svr__epsilon': 0.1, 'svr__gamma': 0.001, 'svr__kernel': 'rbf'}

Model Performance:
- MAE  : 564.5907
- RMSE : 762.2151
- R2   : -1.3152
- Direction Accuracy: 44.60%

Latest Prediction:
- Latest close price: 1949.72
- Predicted next close price: 629.14
- Analyst signal: SELL

